# Robustez de H1 · reanalisis con las condiciones validadas



## 1. Subir los dos ficheros de los modelos

In [ ]:
N_PERMUTACIONES = 10000
SEMILLA = 20260301

# Condiciones que la validacion humana respalda. Bizarre y Change quedan fuera.
CONDICIONES_VALIDADAS = ["Baseline", "Priming", "Confirmation", "Flaws"]

RUTA_GPT = RUTA_LLAMA = None
try:
    from google.colab import files
    print("Sube PRIMERO el fichero de GPT (survey_results_GPT.xlsx)")
    RUTA_GPT = list(files.upload().keys())[0]
    print("\nAhora sube el fichero de LLaMA (survey_results_Ollama.xlsx)")
    RUTA_LLAMA = list(files.upload().keys())[0]
except ImportError:
    RUTA_GPT, RUTA_LLAMA = "survey_results_GPT.xlsx", "survey_results_Ollama.xlsx"

print("\nGPT  :", RUTA_GPT)
print("LLaMA:", RUTA_LLAMA)

## 2. Decodificar la condicion de cada texto



In [ ]:
import pandas as pd
import numpy as np
import re

rng = np.random.default_rng(SEMILLA)
pd.set_option("display.width", 200)

# Patrones de titular, identicos a los del script de limpieza original.
# El orden importa: "Change" se comprueba antes que "Baseline" porque el
# titular de Change en el tema del virus contiene tambien el del baseline.
# Los patrones son deliberadamente literales: un titular truncado o alterado
# no se clasifica, y su set se descarta como no analizable.
PATRONES = [
    # Virus
    ("Change",       r"what we know so far in comparison to the previous variant"),
    ("Baseline",     r"new respiratory virus variant: what we know so far"),
    ("Priming",      r"how it connects to the old respiratory virus"),
    ("Bizarre",      r"young mother admitted to the hospital"),
    ("Confirmation", r"you may already be better prepared than you think"),
    ("Flaws",        r"are people under-reacting to the risks"),
    # Inflacion
    ("Baseline",     r"inflation remains within recent range, new data shows"),
    ("Priming",      r"as repeatedly reported in recent months"),
    ("Bizarre",      r"consumers notice some unusual price changes"),
    ("Change",       r"several goods are up from last month"),
    ("Confirmation", r"some everyday purchases may feel more expensive"),
    ("Flaws",        r"many people misunderstand what the data show"),
]

# Etiquetas ya limpias que aparecen en el fichero de LLaMA
ETIQUETAS = {"BaselineVersion": "Baseline", "BizarreVersion": "Bizarre",
             "ChangedVersion": "Change", "ConfirmationVersion": "Confirmation",
             "FlawsVersion": "Flaws", "PrimedVersion": "Priming"}

# Fallback del script original, para salidas parcialmente limpiadas a mano
FALLBACK = [("change", "Change"), ("base", "Baseline"), ("prim", "Priming"),
            ("bizarre", "Bizarre"), ("confirmation", "Confirmation"),
            ("flaw", "Flaws")]

def condicion_de(valor):
    if pd.isna(valor):
        return None
    s = str(valor).strip()
    if s in ETIQUETAS:
        return ETIQUETAS[s]
    s = s.lower()
    for nombre, patron in PATRONES:
        if re.search(patron, s):
            return nombre
    for clave, nombre in FALLBACK:
        if clave in s:
            return nombre
    return None

def leer(ruta, columnas, modelo):
    d = pd.read_excel(ruta)
    faltan = [c for c in columnas if c not in d.columns]
    if faltan:
        raise ValueError(f"{ruta}: faltan columnas {faltan}. Columnas: {list(d.columns)}")
    filas = []
    for _, r in d.iterrows():
        set_id = f"{modelo} | {r['Order']} | {r['task']} | {r['iteration']}"
        for posicion, col in enumerate(columnas, start=1):
            filas.append({"modelo": modelo, "set_id": set_id,
                          "orden": r["Order"], "tarea": r["task"],
                          "iteracion": r["iteration"],
                          "condicion": condicion_de(r[col]),
                          "rango": posicion})
    return pd.DataFrame(filas)

gpt   = leer(RUTA_GPT,   [f"rank_{i}"      for i in range(1, 7)], "GPT-4o")
llama = leer(RUTA_LLAMA, [f"rank_{i}_f"    for i in range(1, 7)], "LLaMA-3.1-8B")

datos = pd.concat([gpt, llama], ignore_index=True)
datos["tema"] = np.where(datos["orden"].str.startswith("virus"), "Virus", "Inflacion")
datos["autoridad"] = np.where(datos["orden"].str.contains("WITHOUT"),
                              "No authority", "Authority")
print(f"Filas leidas: {len(datos)}  ({datos['set_id'].nunique()} sets)")

## 3. Descartar los sets no analizables



In [ ]:
# Paso 1: fuera las filas cuya condicion no se identifica
datos_ok = datos[datos["condicion"].notna()].copy()

sin_clasificar = len(datos) - len(datos_ok)
print(f"Filas sin condicion identificada: {sin_clasificar}")

# Paso 2: un set es analizable si conserva 6 filas, 6 condiciones y 6 rangos
estado = (datos_ok.groupby(["modelo", "set_id"])
          .agg(filas=("condicion", "size"),
               condiciones=("condicion", "nunique"),
               rangos=("rango", "nunique"))
          .reset_index())
estado["valido"] = ((estado["filas"] == 6) &
                    (estado["condiciones"] == 6) &
                    (estado["rangos"] == 6))

# los sets que perdieron todas sus filas no aparecen en 'estado'
todos = datos[["modelo", "set_id"]].drop_duplicates()
estado = todos.merge(estado, on=["modelo", "set_id"], how="left")
estado[["filas", "condiciones", "rangos"]] = estado[["filas", "condiciones", "rangos"]].fillna(0)
estado["valido"] = estado["valido"].fillna(False)

print("\nSets por modelo:")
print(estado.groupby("modelo")["valido"].agg(validos="sum", total="size").to_string())

excluidos = estado[~estado["valido"]]
if len(excluidos):
    print("\nSets excluidos:")
    display(excluidos[["modelo", "set_id", "filas", "condiciones", "rangos"]]
            .reset_index(drop=True))

val = datos_ok.merge(estado.loc[estado["valido"], ["set_id"]], on="set_id")
print(f"\nSets analizables: {val['set_id'].nunique()}")

## 4. Rangos medios por condicion



In [ ]:
ORDEN = ["Baseline", "Bizarre", "Change", "Confirmation", "Flaws", "Priming"]
display(val.pivot_table(index=["modelo", "tarea"], columns="condicion",
                        values="rango", aggfunc="mean").round(2)[ORDEN])

## 5. El test de permutacion



In [ ]:
def test_permutacion(df, n_perm=N_PERMUTACIONES):
    condiciones = sorted(df["condicion"].unique())
    k = len(condiciones)
    esperado = (k + 1) / 2
    M = np.array([g.set_index("condicion").loc[condiciones, "rango"].values
                  for _, g in df.groupby("set_id")])          # sets x condiciones
    T_obs = ((M.mean(0) - esperado) ** 2).sum()
    idx = rng.random((n_perm,) + M.shape).argsort(axis=2)
    perm = np.take_along_axis(np.broadcast_to(M, (n_perm,) + M.shape), idx, axis=2)
    T_null = ((perm.mean(1) - esperado) ** 2).sum(1)
    p = (np.sum(T_null >= T_obs) + 1) / (n_perm + 1)
    return T_obs, p, M.shape[0], k

def benjamini_hochberg(p):
    p = np.asarray(p, dtype=float)
    o = np.argsort(p)
    r = np.empty_like(o); r[o] = np.arange(1, len(p) + 1)
    adj = p * len(p) / r
    return np.minimum(np.minimum.accumulate(adj[o][::-1])[::-1][np.argsort(o)], 1)

def correr(df, etiqueta):
    filas = []
    for modelo, g_modelo in df.groupby("modelo"):
        sub = []
        for tarea, g in g_modelo.groupby("tarea"):
            T, p, n, k = test_permutacion(g)
            sub.append({"analisis": etiqueta, "modelo": modelo, "tarea": tarea,
                        "k": k, "T": round(T, 2), "p": p, "sets": n})
        for fila, p_bh in zip(sub, benjamini_hochberg([f["p"] for f in sub])):
            fila["p_BH"] = round(p_bh, 4)
            fila["p"] = round(fila["p"], 4)
        filas += sub
    return pd.DataFrame(filas)

completo = correr(val, "6 condiciones")

restringido = (val[val["condicion"].isin(CONDICIONES_VALIDADAS)]
               .copy()
               .assign(rango=lambda d: d.groupby("set_id")["rango"]
                       .rank(method="first")))
restringido = correr(restringido, "4 validadas")

print("Hecho")

## 6. La comparacion


In [ ]:
comp = (pd.concat([completo, restringido])
        .pivot_table(index=["modelo", "tarea"], columns="analisis",
                     values=["T", "p_BH"], aggfunc="first"))
comp = comp[[("T", "6 condiciones"), ("p_BH", "6 condiciones"),
             ("T", "4 validadas"),  ("p_BH", "4 validadas")]]
comp.columns = ["T (6 cond.)", "p_BH (6 cond.)", "T (4 valid.)", "p_BH (4 valid.)"]
comp["mismo veredicto"] = np.where(
    (comp["p_BH (6 cond.)"] < .05) == (comp["p_BH (4 valid.)"] < .05), "si", "NO")
display(comp)

iguales = (comp["mismo veredicto"] == "si").sum()
print(f"{iguales} de {len(comp)} pruebas mantienen el veredicto.")
print()
print("AVISO: los valores de T NO son comparables entre columnas. Al pasar de")
print("seis condiciones a cuatro, el nulo cambia de 3.5 a 2.5 y la escala del")
print("estadistico tambien. Lo comparable es el patron de significacion.")

## 7. Resumen para el manuscrito

In [ ]:
sig6 = comp["p_BH (6 cond.)"] < .05
sig4 = comp["p_BH (4 valid.)"] < .05
print(f"Con las seis condiciones : {sig6.sum()} de {len(comp)} pruebas se apartan de la intercambiabilidad")
print(f"Con las cuatro validadas : {sig4.sum()} de {len(comp)} pruebas")
print()
if (sig6 == sig4).all():
    print("Todas las pruebas mantienen su veredicto. La sensibilidad a la")
    print("presentacion no es atribuible a las condiciones bizarre y change.")
else:
    print("Alguna prueba cambia de veredicto:")
    display(comp[sig6 != sig4])
print()
print("No significativas en ambos analisis:")
for (m, t), r in comp[~sig6].iterrows():
    print(f"  {m} — {t}")